Импорты

In [30]:
!pip install -q torch==2.3.0
!pip install -q pytorch-lightning==2.0.0
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [31]:
!pip install torchtext==0.18.0

In [32]:
!pip install torchvision==0.18.0

In [33]:
import random
import numpy as np
import torch
import pandas as pd
from pathlib import Path
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.nn.functional as F
from torch.optim import AdamW
from torchmetrics import Accuracy
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence

Зафиксируем сиды для воспроизводимости

In [34]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

Загрузка стоп-слов

In [35]:
nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Предобработка данных

In [36]:
def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    text = ' '.join([word for word in text.split() if word not in STOP_WORDS])
    return text

data_path = "/content/tripadvisor_hotel_reviews.csv"
df = pd.read_csv(data_path)
df = df.dropna(subset=['Review', 'Rating'])
df['Review'] = df['Review'].apply(preprocess_text)
X = df['Review'].values
y = df['Rating'].values - 1

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

def yield_tokens(data_iter):
    for text in data_iter:
        yield text.split()

# Исправлено: добавление специального токена <unk>
vocab = build_vocab_from_iterator(yield_tokens(X_train), min_freq=2, specials=["<unk>"])
vocab.set_default_index(0)  # Устанавливаем индекс для неизвестных токенов
vocab_size = len(vocab)

def text_pipeline(text):
    return vocab(text.split())

def create_tensors(texts, labels):
    sequences = [torch.tensor(text_pipeline(text), dtype=torch.long) for text in texts]
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    targets = torch.tensor(labels, dtype=torch.long)
    return padded, targets

train_inputs, train_labels = create_tensors(X_train, y_train)
val_inputs, val_labels = create_tensors(X_val, y_val)
test_inputs, test_labels = create_tensors(X_test, y_test)

BATCH_SIZE = 64

class ReviewDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]

train_loader = DataLoader(ReviewDataset(train_inputs, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(ReviewDataset(val_inputs, val_labels), batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(ReviewDataset(test_inputs, test_labels), batch_size=BATCH_SIZE, num_workers=2)


Сеть с использованием слоев Conv1d

In [37]:
class ConvModel(pl.LightningModule):
    def __init__(self, vocab_size, embed_dim=100, num_classes=5):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, 128, kernel_size=3)
        self.conv2 = nn.Conv1d(embed_dim, 128, kernel_size=4)
        self.conv3 = nn.Conv1d(embed_dim, 128, kernel_size=5)
        self.fc = nn.Sequential(
            nn.Linear(128 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        self.loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float))

    def forward(self, x):
        embedded = self.embedding(x).permute(0, 2, 1)
        conved1 = F.relu(self.conv1(embedded))
        conved2 = F.relu(self.conv2(embedded))
        conved3 = F.relu(self.conv3(embedded))
        pooled1 = F.max_pool1d(conved1, conved1.shape[2]).squeeze(2)
        pooled2 = F.max_pool1d(conved2, conved2.shape[2]).squeeze(2)
        pooled3 = F.max_pool1d(conved3, conved3.shape[2]).squeeze(2)
        cat = torch.cat((pooled1, pooled2, pooled3), dim=1)
        return self.fc(cat)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('test_loss', loss)
        self.log('test_acc', acc)
        return loss

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=5e-4)

Сеть с использованием LSTM-слоя

In [38]:
class LSTMModel(pl.LightningModule):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=5):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        self.loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float))

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(hidden)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('test_loss', loss)
        self.log('test_acc', acc)
        return loss

    def configure_optimizers(self):
        return AdamW(self.parameters(), lr=2e-4)


Улучшенная модель

In [42]:
class EnhancedAttentionLSTMModel(pl.LightningModule):
    def __init__(self, vocab_size, embed_dim=150, hidden_dim=200, num_classes=5):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.bn = nn.BatchNorm1d(hidden_dim * 2)
        self.dropout = nn.Dropout(0.4)

        # Эффективный Attention
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

        self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        self.loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float))

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)

        attn_scores = self.attention(output).squeeze(2)
        attn_weights = F.softmax(attn_scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), output).squeeze(1)
        context = self.bn(context)
        context = self.dropout(context)

        return self.fc(context)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('test_loss', loss)
        self.log('test_acc', acc)
        return loss

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=3e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2, verbose=False
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_acc",
                "interval": "epoch",
                "frequency": 1
            }
        }

Обучение и сравнение

In [43]:
def train_and_evaluate(model, max_epochs=10, model_name="model"):
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        accelerator='auto',
        devices=1 if torch.cuda.is_available() else None,
        callbacks=[
            pl.callbacks.EarlyStopping(monitor="val_loss", patience=3),
            pl.callbacks.ModelCheckpoint(monitor="val_acc", mode="max", save_top_k=1)
        ],
        enable_progress_bar=True
    )

    print(f"\n{'='*50}")
    print(f"Обучение модели: {model_name}")
    print(f"{'='*50}")

    trainer.fit(model, train_loader, val_loader)
    results = trainer.test(model, test_loader)
    return results[0]['test_acc']

In [44]:
conv_model = ConvModel(vocab_size=vocab_size)
conv_acc = train_and_evaluate(conv_model, max_epochs=8, model_name="Conv1D")

lstm_model = LSTMModel(vocab_size=vocab_size)
lstm_acc = train_and_evaluate(lstm_model, max_epochs=8, model_name="LSTM")

enhanced_model = EnhancedAttentionLSTMModel(vocab_size=vocab_size)
enhanced_acc = train_and_evaluate(enhanced_model, max_epochs=12, model_name="Enhanced LSTM")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type               | Params
-------------------------------------------------
0 | embedding | Embedding          | 2.2 M 
1 | conv1     | Conv1d             | 38.5 K
2 | conv2     | Conv1d             | 51.3 K
3 | conv3     | Conv1d             | 64.1 K
4 | fc        | Sequential         | 99.8 K
5 | accuracy  | MulticlassAccuracy | 0     
6 | loss_fn   | CrossEntropyLoss   | 0     
-------------------------------------------------
2.5 M     Trainable params
0         Non-trainable params
2.5 M     Total params
9.864     To


Обучение модели: Conv1D


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: 0it [00:00, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Runningstage.testing    ┃                           ┃
┃          metric           ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5515979528427124     │
│         test_loss         │    3.1223056316375732     │
└───────────────────────────┴───────────────────────────┘

Вывод результатов

In [45]:
print(f"Conv1D Model:   {conv_acc:.4f}")
print(f"LSTM Model:     {lstm_acc:.4f}")
print(f"Enhanced LSTM:  {enhanced_acc:.4f}")

Conv1D Model:   0.5799
LSTM Model:     0.5284
Enhanced LSTM:  0.5516


Сверточная сеть  неожиданно показала лучший результат — 58% точности, обогнав даже улучшенную LSTM с attention. Это говорит о том, что для анализа коротких отзывов важнее локальные паттерны ("отличный завтрак", "шумные соседи"), которые хорошо ловят свертки, а не долгосрочные зависимости. Базовая LSTM справилась хуже всех, вероятно, из-за переобучения на шумных данных. Улучшенная версия с attention дала промежуточный результат, но обучалась дольше. Для реального применения в TripAdvisor я бы выбрал именно сверточную архитектуру — она быстрее, стабильнее и дает лучшее качество на этом типе данных.